# 16 — Synthetic control with multiple outcomes: the 5 tokenizer channels

The collaborator's `scripts/22` and `Docs/scm_methods_presentation.html` map the K = 8
Sentinel-2 band means to Tian, Lee & Panchenko's *Synthetic Controls with Multiple
Outcomes* (one shared simplex weight vector fitted jointly over the stacked K × T0
block, diagonal V with equal total weight per outcome, COVID-style within-unit x outcome
demeaning) and show it beating per-band standard SCM. This corrects the 8-27 report's
section 2: the framework applies when the outcomes share unit-level loadings — her 8
bands do, and so do our 5 tokenizer channels; only the 980 individual parcel coordinates
were a bad outcome set.

Here the same design is run on **K = 5 tokenizer channels** (chip mean per channel,
`chip_mean` representation, chip-mean fill cache), scored with the notebook-11 machinery
(pooled P01–P08 scaler over 60 sites, per-site standardized test RMSE with train
alongside, mean over 10 sites, S1 before S2, P09 and P10):

| arm | weights | preprocessing |
|---|---|---|
| ch_per | one simplex per channel (5 fits) — standard SCM per outcome | pooled z only |
| ch_joint | one shared simplex over 5 channels x T0 — multiple-outcome SC | pooled z only |
| ch_per_dm | one simplex per channel | + within-site x channel demeaning over train, level added back |
| ch_joint_dm | one shared simplex | + within-site x channel demeaning, level added back |

With the z-scored features and equal row counts per channel, the identity weighting in
the joint fit equals the collaborator's "equal total V per outcome". Gate: `ch_joint`
must reproduce the notebook-11 `chip_mean` rows exactly (same scaler, same stacked fit).


In [1]:
import sys
import numpy as np, pandas as pd
sys.path.insert(0, ".")
import panel_lib as pl, panel_repr as pr, panel_align as pa
pd.set_option("display.width", 220)

panel = pl.Panel.from_npz(pl.LATD / "latents_biweekly.npz")
ALL_SITES = sorted(panel.roster["site_id"]); TREAT = panel.treatments; DON = pr.matched_donors(panel)
TESTS = ((9, range(1, 9)), (10, range(1, 10)))

def features_cm(sensor):
    F = {}
    for s in ALL_SITES:
        for q in range(1, 11):
            v = panel.L(s, sensor, q)
            F[(s, q)] = np.full(5, np.nan) if v is None else pr._repr_raw(pr._A(panel, s, sensor, q), "chip_mean")
    return F

FCM = {sen: features_cm(sen) for sen in ("sentinel1", "sentinel2")}

def zfun(F, train_p):
    T = np.stack([F[(s, q)] for s in ALL_SITES for q in train_p])
    mu = np.nanmean(T, axis=0); sd = np.nanstd(T, axis=0, ddof=1)
    sd[~np.isfinite(sd) | (sd == 0)] = 1.0
    return lambda v: (v - mu) / sd

def fit_channels(F, t, train_p, test_q, joint, demean):
    z = zfun(F, train_p); dl = DON[t]
    Yt = np.stack([z(F[(t, q)]) for q in train_p])                       # (T, 5)
    Xd = [np.stack([z(F[(d, q)]) for q in train_p]) for d in dl]         # J x (T, 5)
    yte = z(F[(t, test_q)]); xte = np.column_stack([z(F[(d, test_q)]) for d in dl])  # (5, J)
    mt = np.nanmean(Yt, 0); md = np.column_stack([np.nanmean(X, 0) for X in Xd])     # (5,), (5, J)
    Ytf = Yt - mt if demean else Yt
    Xdf = [X - md[:, j] for j, X in enumerate(Xd)] if demean else Xd
    if joint:
        w = pa.simplex_scm(Ytf.ravel(), np.column_stack([X.ravel() for X in Xdf]))
        W = np.tile(w, (5, 1)).T                                         # (J, 5)
    else:
        W = np.column_stack([pa.simplex_scm(Ytf[:, k], np.column_stack([X[:, k] for X in Xdf]))
                             for k in range(5)])                         # (J, 5)
    if demean:
        pred_te = mt + np.array([np.nansum(W[:, k] * (xte[k] - md[k])) for k in range(5)])
        pred_tr = mt + np.stack([[np.nansum(W[:, k] * (np.array([X[i, k] for X in Xd]) - md[k]))
                                  for k in range(5)] for i in range(len(train_p))])
    else:
        pred_te = np.array([np.nansum(W[:, k] * xte[k]) for k in range(5)])
        pred_tr = np.stack([[np.nansum(W[:, k] * np.array([X[i, k] for X in Xd])) for k in range(5)]
                            for i in range(len(train_p))])
    te, tr = pa.rmse(yte - pred_te), pa.rmse((Yt - pred_tr).ravel())
    c2, c3 = pa.rmse(yte - mt), pa.rmse(yte - np.nanmean(xte, 1))
    return {"site": t, "test": f"P{test_q:02d}", "test_rmse": te, "train_rmse": tr,
            "own_hist_rmse": c2, "equal_w_rmse": c3, "C1": te <= 1.5 * tr, "C2": te < c2,
            "C3": te < c3, "w_max": float(np.nanmax(W))}

def summarize(df):
    g = df.groupby(["sensor", "arm", "test"])
    out = g[["test_rmse", "train_rmse"]].mean().round(3)
    out["C1"] = g.C1.sum(); out["C2"] = g.C2.sum(); out["C3"] = g.C3.sum(); out["n"] = g.size()
    return out
print("ready")


ready


## 1. The four channel arms, gate against notebook 11

In [2]:
ARMS = {"ch_per": (False, False), "ch_joint": (True, False),
        "ch_per_dm": (False, True), "ch_joint_dm": (True, True)}
rows = []
for sen in ("sentinel1", "sentinel2"):
    for arm, (joint, demean) in ARMS.items():
        for test_q, train_p in TESTS:
            for t in TREAT:
                r = fit_channels(FCM[sen], t, train_p, test_q, joint, demean)
                r.update(arm=arm, sensor=sen); rows.append(r)
sites = pd.DataFrame(rows)

ref = pd.read_csv("panel_collabstyle_scm_validation.csv").query("cache == 'chipmean' and repr == 'chip_mean'")
got = sites.query("arm == 'ch_joint'").groupby(["sensor", "test"]).test_rmse.mean()
for _, r in ref.iterrows():
    g = got[(r.sensor, r.test)]
    assert abs(g - r.mean_test_rmse) < 1e-6, (r.sensor, r.test, g, r.mean_test_rmse)
print("gate: ch_joint == notebook-11 chip_mean rows (max |diff| "
      f"{max(abs(got[(r.sensor, r.test)] - r.mean_test_rmse) for _, r in ref.iterrows()):.2e})")
print()
print(summarize(sites).to_string())


gate: ch_joint == notebook-11 chip_mean rows (max |diff| 0.00e+00)

                            test_rmse  train_rmse  C1  C2  C3   n
sensor    arm         test                                       
sentinel1 ch_joint    P09       0.603       0.620  10   4   7  10
                      P10       0.701       0.624   9   4  10  10
          ch_joint_dm P09       0.513       0.398   5   8   8  10
                      P10       0.385       0.407   9   9  10  10
          ch_per      P09       0.495       0.486   9   8   9  10
                      P10       0.625       0.491   6   4   8  10
          ch_per_dm   P09       0.519       0.348   4   7   8  10
                      P10       0.447       0.364   9   7   9  10
sentinel2 ch_joint    P09       0.577       0.741  10   8   3  10
                      P10       0.395       0.736  10   9   7  10
          ch_joint_dm P09       0.711       0.648   8   6   2  10
                      P10       0.380       0.661  10  10   7  10
        

## 2. Exploratory: the same demeaning on the 980-d latent (whole image)

The collaborator's Exp-1-vs-Exp-2 gap shows the within-unit demeaning is load-bearing.
Does it also help when the outcome set is the whole image? Same machinery at 980-d
(per-channel scaler as in notebook 13, since the pooled per-dimension scaler is
position-specific): identity positions (arm A) and the class + TerraMind-history
alignment (arm D_min from notebook 12), each with and without within-site x coordinate
demeaning. Notebook-13 baselines: A (perdim scaler) S1 1.08, D_min 0.79.


In [3]:
DESC = pa.load_descriptors("parcel_descriptors.npz")
_z = np.load("parcel_perms.npz"); PERMS = {tuple(k.split("|")): _z[k] for k in _z.files}
IDENT = np.arange(196)

def pcz(sensor, train_p):
    T = np.stack([panel.L(s, sensor, q) for s in ALL_SITES for q in train_p
                  if panel.L(s, sensor, q) is not None])
    V = T.reshape(len(T), 5, 196).transpose(1, 0, 2).reshape(5, -1)
    mu, sd = np.repeat(V.mean(1), 196), np.repeat(V.std(1, ddof=1), 196)
    sd[~np.isfinite(sd) | (sd == 0)] = 1.0
    return lambda v: (v - mu) / sd

def vec(site, sensor, q, perm=None):
    if perm is None:
        v = panel.L(site, sensor, q)
        return np.full(980, np.nan) if v is None else v
    return pa.aligned_vec(panel, site, sensor, q, perm)

def fit_latent(sensor, t, train_p, test_q, aligned, demean):
    z = pcz(sensor, train_p)
    perm_of = {}
    for j in DON[t]:
        if not aligned:
            perm_of[j] = None            # identity read, no NaN masking
        else:
            p = PERMS[("D_min", sensor, t, j)]
            perm_of[j] = p if (p >= 0).sum() >= pa.MIN_MATCHED else "drop"
    dl = [j for j in DON[t] if not (isinstance(perm_of[j], str))]
    Yt = np.stack([z(vec(t, sensor, q)) for q in train_p])
    Xd = [np.stack([z(vec(j, sensor, q, perm_of[j])) for q in train_p]) for j in dl]
    yte = z(vec(t, sensor, test_q)); xte = np.column_stack([z(vec(j, sensor, test_q, perm_of[j])) for j in dl])
    mt = np.nanmean(Yt, 0); md = np.column_stack([np.nanmean(X, 0) for X in Xd])
    Ytf = Yt - mt if demean else Yt
    Xdf = [X - md[:, j] for j, X in enumerate(Xd)] if demean else Xd
    w = pa.simplex_scm(Ytf.ravel(), np.column_stack([X.ravel() for X in Xdf]))
    if demean:
        pred_te = mt + (xte - md) @ w
        pred_tr = mt + np.stack([sum(w[j] * (Xd[j][i] - md[:, j]) for j in range(len(dl)))
                                 for i in range(len(train_p))])
    else:
        pred_te = xte @ w
        pred_tr = np.stack([sum(w[j] * Xd[j][i] for j in range(len(dl))) for i in range(len(train_p))])
    te, tr = pa.rmse(yte - pred_te), pa.rmse((Yt - pred_tr).ravel())
    c2, c3 = pa.rmse(yte - mt), pa.rmse(yte - np.nanmean(xte, 1))
    return {"site": t, "test": f"P{test_q:02d}", "test_rmse": te, "train_rmse": tr,
            "own_hist_rmse": c2, "equal_w_rmse": c3, "C1": te <= 1.5 * tr, "C2": te < c2,
            "C3": te < c3, "w_max": float(w.max()), "n_donors": len(dl)}

LARMS = {"lat_A": (False, False), "lat_A_dm": (False, True),
         "lat_Dmin": (True, False), "lat_Dmin_dm": (True, True)}
lrows = []
for sen in ("sentinel1", "sentinel2"):
    for arm, (aligned, demean) in LARMS.items():
        for test_q, train_p in TESTS:
            for t in TREAT:
                r = fit_latent(sen, t, train_p, test_q, aligned, demean)
                r.update(arm=arm, sensor=sen); lrows.append(r)
lsites = pd.DataFrame(lrows)
print(summarize(lsites).to_string())


/tmp/ipykernel_3619844/611521037.py:32: RuntimeWarning: Mean of empty slice
  mt = np.nanmean(Yt, 0); md = np.column_stack([np.nanmean(X, 0) for X in Xd])
/tmp/ipykernel_3619844/611521037.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - mt), pa.rmse(yte - np.nanmean(xte, 1))


                            test_rmse  train_rmse  C1  C2  C3   n
sensor    arm         test                                       
sentinel1 lat_A       P09       1.079       1.088  10   0  10  10
                      P10       1.082       1.088  10   0   7  10
          lat_A_dm    P09       0.750       0.676  10   0  10  10
                      P10       0.754       0.679  10   0  10  10
          lat_Dmin    P09       0.791       0.734  10   0   9  10
                      P10       0.797       0.742  10   0  10  10
          lat_Dmin_dm P09       0.758       0.688  10   1  10  10
                      P10       0.757       0.691  10   2  10  10
sentinel2 lat_A       P09       1.091       0.944  10   1   4  10
                      P10       1.234       0.951  10   0   2  10
          lat_A_dm    P09       1.049       0.788  10   1   8  10
                      P10       1.004       0.803  10   3  10  10
          lat_Dmin    P09       0.992       0.813  10   5   7  10
          

In [4]:
allsites = pd.concat([sites, lsites], ignore_index=True)
allsites.to_csv("panel_scmmo_channels_sites.csv", index=False)
s = summarize(allsites).reset_index()
s.to_csv("panel_scmmo_channels_validation.csv", index=False)
print("saved panel_scmmo_channels_sites.csv, panel_scmmo_channels_validation.csv")


saved panel_scmmo_channels_sites.csv, panel_scmmo_channels_validation.csv


## Reading

1. **The gate holds**: `ch_joint` (one shared simplex over 5 channels × T0, pooled z) is
   *identical* to the notebook-11 `chip_mean` rows (max diff 0.00e+00) — notebook 11 was
   already a multiple-outcome SC in Tian–Lee–Panchenko form; what it lacked was the
   within-unit demeaning.
2. **Demeaning is load-bearing on Sentinel-1**: `ch_joint_dm` 0.513 / **0.385** vs
   `ch_joint` 0.603 / 0.701 — a 45 % cut at P10, the best S1 pooled number in the
   project so far, with C2 8–9/10 (it now beats the site's own history, which no
   cross-sectional S1 arm did before). The joint fit also beats the per-channel fits
   under demeaning at P10 (0.385 vs 0.447), reproducing the collaborator's
   multi-outcome > single-outcome finding on our channels.
3. **Sentinel-2 is flatter**: demeaning helps only at P10 (`ch_per_dm` **0.369**,
   `ch_joint_dm` 0.380, vs 0.395 nb11) and hurts at P09 (0.65–0.71) — P09 S2 chips are
   cloud-heavy, so train-window level estimates are noisy and the level-add-back
   transfers that noise.
4. **On the whole image (980-d), demeaning substitutes for alignment on S1**: identity
   positions + demeaning (`lat_A_dm` 0.750/0.754) ≈ Hungarian class+history alignment
   (`lat_Dmin` 0.791/0.797), and combining both adds nothing (0.758/0.757). Removing
   each site's own spatial mean image is doing the same job as re-arranging donor
   parcels: what is left to match is temporal dynamics, which are position-free.
5. **On S2 the two are not substitutes**: alignment matters (`lat_Dmin` 0.99/0.90 vs
   `lat_A_dm` 1.05/1.00), demeaning adds a little only at P10 (`lat_Dmin_dm` 0.887).
6. Own-history mean (C2 comparator) still beats every cross-sectional 980-d arm on S1
   (C2 ≤ 2/10) — consistent with Experiment 2's finding that S1 dynamics are essentially
   a site-specific constant plus noise.
